## Table of Contents
* [Frontmatter] (#frontmatter)
* [Abstract](#abstract)
* [Introduction](#intro)
* [Data Cleaning](#data-cleaning)
* [Results](#results)


## Frontmatter <a id="frontmatter"></a>

Diabetes Readmission Predictions Using Machine Learning 
Team: Jayden Chen, Evan Lin, Jonathan Mota
Repo URL: https://github.com/JayBirddy/Middlebury-ML-Class-Proj

## Abstract <a id="abstract"></a>

Your abstract is a one-paragraph summary of the problem you addressed, the approach(es) that you used to address it, and the big-picture results that you obtained.

Hospital readmission prediction represents a significant burden, not only on the individual, but also on the health institutions. From the patient's perspective, readmission predictions can make the difference between effective treatment or prolonged health issues, whilst from a hospital or a companies standpoint it may result in wasted resources or financial losses. Prior work has demonstrated the feasibility of predicting readmission rates within a 30-day period using structured electronic health record data. Namely, one such prominent dataset that has been the subject of a number of research studies is the "Diabetes 130-US Hospitals for Years 1999-2008" dataset. In this project, we train and compare two main binary classifiers: stochastic logistic regression, a basic implementation of a multilayer perception neural networks, as well as two other neural networks with various feature engineerings to see how results could be effected. 

## Introduction <a id="introduction"></a>

Necessary imports

In [23]:
import pandas as pd
import pickle
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import roc_auc_score

Data exploration

In [ ]:
# read the dataset into dataframe
data = pd.read_csv("data/diabetic_data.csv")

In [5]:
# see the size of the dataset 
data.shape

(101766, 50)

In [6]:
# check out data-types
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype
---  ------                    --------------   -----
 0   encounter_id              101766 non-null  int64
 1   patient_nbr               101766 non-null  int64
 2   race                      101766 non-null  str  
 3   gender                    101766 non-null  str  
 4   age                       101766 non-null  str  
 5   weight                    101766 non-null  str  
 6   admission_type_id         101766 non-null  int64
 7   discharge_disposition_id  101766 non-null  int64
 8   admission_source_id       101766 non-null  int64
 9   time_in_hospital          101766 non-null  int64
 10  payer_code                101766 non-null  str  
 11  medical_specialty         101766 non-null  str  
 12  num_lab_procedures        101766 non-null  int64
 13  num_procedures            101766 non-null  int64
 14  num_medications           10176

In [30]:
second_row = data.iloc[1]  # returns the 2nd row as a Series
display(second_row)

encounter_id                   149190
patient_nbr                  55629189
race                        Caucasian
gender                         Female
age                           [10-20)
weight                              ?
admission_type_id                   1
discharge_disposition_id            1
admission_source_id                 7
time_in_hospital                    3
payer_code                          ?
medical_specialty                   ?
num_lab_procedures                 59
num_procedures                      0
num_medications                    18
number_outpatient                   0
number_emergency                    0
number_inpatient                    0
diag_1                            276
diag_2                         250.01
diag_3                            255
number_diagnoses                    9
max_glu_serum                     NaN
A1Cresult                         NaN
metformin                          No
repaglinide                        No
nateglinide 

In [7]:
# check numerical data values 
data.describe().transpose()

,count,mean,std,min,25%,50%,75%,max
encounter_id,101766.0,1.652016e+08,1.026403e+08,12522.0,84961194.0,152388987.0,2.302709e+08,443867222.0
patient_nbr,101766.0,5.433040e+07,3.869636e+07,135.0,23413221.0,45505143.0,8.754595e+07,189502619.0
admission_type_id,101766.0,2.024006e+00,1.445403e+00,1.0,1.0,1.0,3.000000e+00,8.0
discharge_disposition_id,101766.0,3.715642e+00,5.280166e+00,1.0,1.0,1.0,4.000000e+00,28.0
admission_source_id,101766.0,5.754437e+00,4.064081e+00,1.0,1.0,7.0,7.000000e+00,25.0
time_in_hospital,101766.0,4.395987e+00,2.985108e+00,1.0,2.0,4.0,6.000000e+00,14.0
num_lab_procedures,101766.0,4.309564e+01,1.967436e+01,1.0,31.0,44.0,5.700000e+01,132.0
num_procedures,101766.0,1.339730e+00,1.705807e+00,0.0,0.0,1.0,2.000000e+00,6.0
num_medications,101766.0,1.602184e+01,8.127566e+00,1.0,10.0,15.0,2.000000e+01,81.0
number_outpatient,101766.0,3.693572e-01,1.267265e+00,0.0,0.0,0.0,0.000000e+00,42.0


In [27]:
# make a copy of the file for pre-processing 
train = data.copy(deep=True)

In [ ]:
# if '?' is still a string
train.replace('?', np.nan, inplace=True)

# check missing values
missing_value = (
    train.isnull().mean()
      .mul(100)
      .round(2)
      .reset_index()
      .rename(columns={"index": "col", 0: "percent_missing"})
      .sort_values("percent_missing", ascending=False)
)
missing_value = missing_value[missing_value["percent_missing"] > 0]
display(missing_value)


,col,percent_missing
5,weight,96.86
22,max_glu_serum,94.75
23,A1Cresult,83.28
11,medical_specialty,49.08
10,payer_code,39.56
2,race,2.23
20,diag_3,1.40
19,diag_2,0.35
18,diag_1,0.02


In [ ]:
# drop the irrelavant and high missing value variables
# weight (97% missing), medical_specialty (49.8% missing), payer_code (40.5% missing)
train=train.drop(['weight','medical_specialty', 'payer_code'],axis=1)

# drop only missing values in all three diagonosis categories 
train = train.drop(set(train[(train['diag_1']== '?') & (train['diag_2'] == '?') & (train['diag_3'] == '?')].index))

# drop patients who passed after discharge due to non relevant to the readmission prediction task
train = train.drop(set(train[train['discharge_disposition_id']==11].index))

train.shape

(100123, 47)